# 4 · Hooks: conception

A **hook** is a callback the agent invokes at a defined lifecycle boundary.
Hooks *observe* — they log, they gate, they record telemetry — without being
part of the decision-making core. Think of them as the "spies" around the loop.

## The six hook events

| Hook | Fired when |
|------|-----------|
| `pretooluse`   | just *before* a tool runs |
| `posttooluse`  | just *after* a tool returns its result |
| `userpromptsubmit` | when the user's message enters the loop |
| `stop`         | when the turn finishes (`finish_reason = stop`) |
| `subagentstart`| when a sub-agent is spawned |
| `subagentstop` | when a sub-agent finishes |

## How the agent incorporates hooks

The loop calls `hooks.fire(event, payload)` at exactly six places. In this
project those call sites live in `src/agent.rs`:

- `UserPromptSubmit` — at the very top of `Agent::run`.
- `PreToolUse` — once per tool call, *before* execution.
- `PostToolUse` — once per tool call, *after* its result is produced.
- `Stop` — right before `run` returns the final answer.
- `SubAgentStart` / `SubAgentStop` — bracketing `run_subagent`.

The core loop does **not** change based on which hooks are registered; hooks are
an additive side-channel. Let's inspect that mechanism directly.


In [ ]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

use agent_loop::hooks::{Hooks, HookEvent, HookPayload};
use std::sync::{Arc, Mutex};

// A Hooks registry with a shared log buffer enabled.
let mut hooks = Hooks::new().with_log();

// Attach one callback per event. Each prints a distinguishable line.
hooks.on_pre_tool_use(        |p| println!("  [handle {:<4}] {}", "PRE",  label(p)));
hooks.on_post_tool_use(       |p| println!("  [handle {:<4}] {}", "POST", label(p)));
hooks.on_user_prompt_submit(  |p| println!("  [handle {:<12}] {}", "USER", label(p)));
hooks.on_stop(                |p| println!("  [handle {:<8}] {}", "STOP", label(p)));
hooks.on_sub_agent_start(     |p| println!("  [handle {:<8}] {}", "SSTART", label(p)));
hooks.on_sub_agent_stop(      |p| println!("  [handle {:<8}] {}", "SSTOP", label(p)));

// Small helper to render any payload headline.
fn label(p: &HookPayload) -> String {
    match p {
        HookPayload::PreToolUse { tool, .. } => format!("about to call `{tool}`"),
        HookPayload::PostToolUse { tool, is_error, .. } => format!("`{tool}` finished, is_error={is_error}"),
        HookPayload::UserPromptSubmit { prompt, .. } => format!("prompt: {:?}", &prompt[..prompt.len().min(20)]),
        HookPayload::Stop { reason } => format!("turn over ({reason})"),
        HookPayload::SubAgentStart { name, .. } => format!("spawn {name}"),
        HookPayload::SubAgentStop { name, .. } => format!("finished {name}"),
    }
}

println!("registered callbacks: {}", hooks.count());
println!("supported events    : {:?}", HookEvent::ALL.iter().map(|e| e.as_str()).collect::<Vec<_>>());


## Registry ↔ agent wiring

The loop only ever calls two methods:

- `hooks.fire(event, payload)` — run every callback for that event **and**
  append to the shared log (when enabled);
- `hook_name()` / the six `on_*` builders — how you subscribe.

Because callbacks are just closures, multiple hooks can subscribe to one event,
and you add/remove them dynamically — the same spirit as dynamic tools. Demo 5
fires each of the six for real and shows the ordered log.
